# A Multi-Class Text Classification Model with Naive Bayes

It's pretty cool, if you stop to think about it, that gmail -- or your email service of choice -- automatically knows whether a message is spam, a promotion, or something important. There are all sorts of folders or *classes* in which an email could land.

Behind that magic is a type of machine learning called *multi-class text classification*.

This notebook builds a multi-class model to do just that -- classify text into specific domains. But instead of filtering emails, we're predicting which **scientific field** a research article belongs to, using its abstract and title.

The algorithm we use is called **Naive Bayes**, a simple yet powerful method that calculates the probability a piece of text belongs to each category based on the words it contains. We'll explore exactly how it works later in the notebook-- but for now, think of it as a fast, interpretable way to make predictions from text.

Let's begin with a roadmap of what is to come and then a quick look at the data.

## Table of Contents
- The Dataset
- Import pandas and load the dataset
- Assigning categories with an elegant for loop
- Split the data for training and testing
- Clean the Text for Vectorization
- Initialize the vectorizer
- Build the document-term matrices
- Train the model with Naive Bayes
- Make predictions and understand the output
- Evaluate the model's performance

## The Dataset

This dataset, which is titled [Multi-Label Classification Dataset](https://www.kaggle.com/datasets/shivanandmn/multilabel-classification-dataset?resource=download), comes from Kaggle. Here's the official description of what it includes:

"The dataset contains 6 different labels (Computer Science, Physics, Mathematics, Statistics, Quantitative Biology, Quantitative Finance) to classify the research papers based on Abstract and Title.

The value 1 in label columns represents that label belongs to that paper. Each paper has multiple labels as 1."

## Import pandas and load the dataset

In [3]:
import pandas as pd

In [5]:
# Load the dataset
# Note: Each article has a '1' in just one of the six columns
articles = pd.read_csv('articles.csv')
articles.head()

,ID,TITLE,ABSTRACT,Computer Science,Physics,Mathematics,Statistics,Quantitative Biology,Quantitative Finance
0,1,Reconstructing Subject-Specific Effect Maps,Predictive models allow subject-specific inf...,1,0,0,0,0,0
1,2,Rotation Invariance Neural Network,Rotation invariance and translation invarian...,1,0,0,0,0,0
2,3,Spherical polyharmonics and Poisson kernels fo...,We introduce and develop the notion of spher...,0,0,1,0,0,0
3,4,A finite element approximation for the stochas...,The stochastic Landau--Lifshitz--Gilbert (LL...,0,0,1,0,0,0
4,5,Comparative study of Discrete Wavelet Transfor...,Fourier-transform infra-red (FTIR) spectra o...,1,0,0,1,0,0


In [7]:
# Check the length
len(articles)

20972

In [14]:
# And check the shape
articles.shape

(20972, 9)

We’re going to make two quick tweaks here:

1. Combine the title and abstract into a single column called 'Text' -- that’ll be our input.
2. Create a 'Category'` column to hold the article’s label (we’ll fill it in next).

This sets us up for modeling with clean text and a single target column.

In [16]:
# Transform the dataset to just two columns or text and class
# Start by creating a new column 'text' that concatenates title and abstract
articles['Text'] = articles['TITLE'] + ' ' + articles['ABSTRACT']

In [18]:
# Check the output
articles.head()

,ID,TITLE,ABSTRACT,Computer Science,Physics,Mathematics,Statistics,Quantitative Biology,Quantitative Finance,Text
0,1,Reconstructing Subject-Specific Effect Maps,Predictive models allow subject-specific inf...,1,0,0,0,0,0,Reconstructing Subject-Specific Effect Maps ...
1,2,Rotation Invariance Neural Network,Rotation invariance and translation invarian...,1,0,0,0,0,0,Rotation Invariance Neural Network Rotation ...
2,3,Spherical polyharmonics and Poisson kernels fo...,We introduce and develop the notion of spher...,0,0,1,0,0,0,Spherical polyharmonics and Poisson kernels fo...
3,4,A finite element approximation for the stochas...,The stochastic Landau--Lifshitz--Gilbert (LL...,0,0,1,0,0,0,A finite element approximation for the stochas...
4,5,Comparative study of Discrete Wavelet Transfor...,Fourier-transform infra-red (FTIR) spectra o...,1,0,0,1,0,0,Comparative study of Discrete Wavelet Transfor...


In [20]:
# Transform the six label columns into one
# Start by creating the new column 'Category' and setting it to 'none'; just need it to be some string!
articles['Category'] = 'none'

In [22]:
# Check the results
articles.head()

,ID,TITLE,ABSTRACT,Computer Science,Physics,Mathematics,Statistics,Quantitative Biology,Quantitative Finance,Text,Category
0,1,Reconstructing Subject-Specific Effect Maps,Predictive models allow subject-specific inf...,1,0,0,0,0,0,Reconstructing Subject-Specific Effect Maps ...,none
1,2,Rotation Invariance Neural Network,Rotation invariance and translation invarian...,1,0,0,0,0,0,Rotation Invariance Neural Network Rotation ...,none
2,3,Spherical polyharmonics and Poisson kernels fo...,We introduce and develop the notion of spher...,0,0,1,0,0,0,Spherical polyharmonics and Poisson kernels fo...,none
3,4,A finite element approximation for the stochas...,The stochastic Landau--Lifshitz--Gilbert (LL...,0,0,1,0,0,0,A finite element approximation for the stochas...,none
4,5,Comparative study of Discrete Wavelet Transfor...,Fourier-transform infra-red (FTIR) spectra o...,1,0,0,1,0,0,Comparative study of Discrete Wavelet Transfor...,none


## Assigning categories with an elegant for loop

Here’s the linchpin.

We loop across the six topic columns and assign the correct label to each row in a single, unified 'Category' column.  
This approach keeps the code flexible and scalable -- no hardcoding, no manual mappings.

It’s a small block of code that brings everything together for modeling.

In [24]:
# Create the for loop 
# Start by excluding columns 3 to 9, the six categories
for c in articles.columns[3:9]:
    print(c)

Computer Science
Physics
Mathematics
Statistics
Quantitative Biology
Quantitative Finance


In [26]:
# Now find the articles, for instance, where 'Computer Science' is equal to 1
c = 'Computer Science'
articles.loc[articles[c] == 1, 'Category'] = c

In [28]:
# Now going to put this back into our function
for c in articles.columns[3:9]:
    print(c)
    articles.loc[articles[c] == 1, 'Category'] = c

Computer Science
Physics
Mathematics
Statistics
Quantitative Biology
Quantitative Finance


In [28]:
# Check the DataFrame to confirm it worked
# Cool! Notice that all the 'nones' in the 'Category' column have been replaced
articles

,ID,TITLE,ABSTRACT,Computer Science,Physics,Mathematics,Statistics,Quantitative Biology,Quantitative Finance,Text,Category
0,1,Reconstructing Subject-Specific Effect Maps,Predictive models allow subject-specific inf...,1,0,0,0,0,0,Reconstructing Subject-Specific Effect Maps ...,Computer Science
1,2,Rotation Invariance Neural Network,Rotation invariance and translation invarian...,1,0,0,0,0,0,Rotation Invariance Neural Network Rotation ...,Computer Science
2,3,Spherical polyharmonics and Poisson kernels fo...,We introduce and develop the notion of spher...,0,0,1,0,0,0,Spherical polyharmonics and Poisson kernels fo...,none
3,4,A finite element approximation for the stochas...,The stochastic Landau--Lifshitz--Gilbert (LL...,0,0,1,0,0,0,A finite element approximation for the stochas...,none
4,5,Comparative study of Discrete Wavelet Transfor...,Fourier-transform infra-red (FTIR) spectra o...,1,0,0,1,0,0,Comparative study of Discrete Wavelet Transfor...,Computer Science
...,...,...,...,...,...,...,...,...,...,...,...
20967,20968,Contemporary machine learning: a guide for pra...,Machine learning is finding increasingly bro...,1,1,0,0,0,0,Contemporary machine learning: a guide for pra...,Computer Science
20968,20969,Uniform diamond coatings on WC-Co hard alloy c...,Polycrystalline diamond coatings have been g...,0,1,0,0,0,0,Uniform diamond coatings on WC-Co hard alloy c...,none
20969,20970,Analysing Soccer Games with Clustering and Con...,We present a new approach for identifying si...,1,0,0,0,0,0,Analysing Soccer Games with Clustering and Con...,Computer Science
20970,20971,On the Efficient Simulation of the Left-Tail o...,The sum of Log-normal variates is encountere...,0,0,1,1,0,0,On the Efficient Simulation of the Left-Tail o...,none


Now to delete all the columns aside from those last two/

In [30]:
# Use .drop(columns=) to remove the first nine columns
# And then overwrite the DataFrame
articles  = articles.drop(columns=articles.columns[:9])
articles.head()

,Text,Category
0,Reconstructing Subject-Specific Effect Maps ...,Computer Science
1,Rotation Invariance Neural Network Rotation ...,Computer Science
2,Spherical polyharmonics and Poisson kernels fo...,none
3,A finite element approximation for the stochas...,none
4,Comparative study of Discrete Wavelet Transfor...,Computer Science


In [32]:
# Check to make sure the 'Category' column doesn't have any 'none' values left
# Use .unique() to display all the distinct elements in the 'Category' column or Series
articles.Category.unique()

array(['Computer Science', 'none'], dtype=object)

## Split the data for training and testing

The setup is done -- now we split the data.

We'll break our cleaned dataset into training and test sets (70-30 split). This lets us train the model on most of the data, then see how well it performs on examples it's never seen before.

In [37]:
from sklearn.model_selection import train_test_split
# So, train_test_split needs the independent variable and the dependent variable
# or articles.Text and articles.Category, set to random state, 
# set the split to 70-30 and set it equal to X_train, X_test 
# (the independent variables) and then
# y_train and y_test the dependent variables
# As we see: This returns the four objects stated above
X_train, X_test, y_train, y_test = train_test_split(articles.Text, articles.Category, random_state=1, test_size=0.3)

In [39]:
# Check the shape of X_train
X_train.shape

(14680,)

In [41]:
# Of course, X_test is the remaining 6,292 per the 70-30 split
X_test.shape

(6292,)

In [43]:
# Notice this, of course, equals the total number of rows in the dataset
X_train.shape[0] + X_test.shape[0]

20972

In [45]:
# And check the shape of y_train
y_train.shape

(14680,)

In [47]:
# And, finally, y_test
y_test.shape

(6292,)

In [51]:
# Display X_train
X_train

11342    Twitter and the Press: an Ego-Centred Analysis...
13834    Refining Trace Abstraction using Abstract Inte...
8439     Predicting Adversarial Examples with High Conf...
2201     Characterization of Thermal Neutron Beam Monit...
9366     Diversity, Topology, and the Risk of Node Re-i...
                               ...                        
10955    Generating large misalignments in gapped and b...
17289    Koszul cycles and Golod rings   Let $S$ be the...
5192     The meet operation in the imbalance lattice of...
12172    Unsupervised learning of object frames by dens...
235      An automata group of intermediate growth and e...
Name: Text, Length: 14680, dtype: object

In [55]:
# Check the type
# For now it's just a pandas Series
type(X_train)

pandas.core.series.Series

In [57]:
# Notice the indicies: Each item has the corresponding label
y_train

11342    Computer Science
13834    Computer Science
8439                 none
2201                 none
9366     Computer Science
               ...       
10955                none
17289                none
5192                 none
12172    Computer Science
235                  none
Name: Category, Length: 14680, dtype: object

In [59]:
# Same for y_test
y_test

14348                none
20220    Computer Science
11721                none
5500                 none
4696     Computer Science
               ...       
5555     Computer Science
777      Computer Science
16130                none
18535                none
3201     Computer Science
Name: Category, Length: 6292, dtype: object

## Clean the text for vectorization

This is the engine room of the mode.

Before we vectorize, we need to clean up the text.

Here’s a simple text preprocessing function that:
- Tokenizes each sentence
- Removes stopwords
- Lemmatizes each token
- Joins the cleaned tokens back into a single string

This ensures that our TF-IDF matrix is built on consistent, meaningful text features.

And as a quick note: This function is very much customizable! Feel free to swap in stemming, expand stopword lists, or add regex cleaning depending on your use case.

In [64]:
# Drop in the 'clean_text' function
from gensim.utils import tokenize
import nltk
nltk.download('punkt')
nltk.download('stopwords')
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

def clean_text(text):
    lemmatizer = WordNetLemmatizer()
    stemmer = PorterStemmer()
    tokens = list(tokenize(text))
    #res = ' '.join([stemmer.stem(t.lower()) for t in tokens if t.lower() not in stop_words]) 
    res = ' '.join([lemmatizer.lemmatize(t.lower()) for t in tokens if t.lower() not in stop_words]) 
    if len(res) == 0:
        return ' '
    else:
        return res

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/karlbuscheck/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/karlbuscheck/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Let’s test it on a sample sentence to make sure it’s working.

In [71]:
# Test out the function
clean_text('Hello from the Leavey School of Business! This text is clean, and brief.')

'hello leavey school business text clean brief'

## Initialize the vectorizer

In [81]:
from sklearn.feature_extraction.text import CountVectorizer

In [83]:
# Intialize the CountVectortizer
# In this case, the corpus, so to speak, is only X_train, but it needs to be cleaned
# So, use the above function with the CountVectorizer using preprocessor and ngram
count_vect = CountVectorizer(preprocessor=clean_text, ngram_range=(1,2))
count_vect.fit(X_train)

CountVectorizer(ngram_range=(1, 2),
                preprocessor=<function clean_text at 0x1760c9440>)

## Build the document-term matrices

Now that the vectorizer is initialized, we’ll apply it to both the training and test sets.

This gives us large, sparse matrices where each row is an article and each column represents a word or phrase from the vocabulary. These matrices will be the actual input to our Naive Bayes classifier.

In [98]:
# Transform X_train to an X_train matrix
# So, this is building the document-term matrix for the training set, which
# will be the actual training set that will be used in the classification task
X_train_mat = count_vect.transform(X_train)

In [99]:
# Check the shape of the matrix
# Note: It is a huge matrix! 14K+ rows and almost 900K columns
X_train_mat.shape

(14680, 879982)

In [100]:
# Now the same for the test set
X_test_mat = count_vect.transform(X_test)

In [101]:
# Now check the shape
# So, as we'd expect based on the 70-30 split,
# the test set has the same number of columns but just over 6K rows!
X_test_mat.shape

(6292, 879982)

## Train the model with Naive Bayes

**Naive Bayes** is a fast, simple, and surprisingly effective classification algorithm based on **Bayes' Theorem**.

It works by calculating the probability that a piece of text belongs to each possible class -- like "Computer Science" or "Physics" -- based on the words it contains. The *naive* part comes from the assumption that all features (in this case, words) are independent of each other. That assumption is rarely true in practice, but the model still performs impressively well in many text classification tasks.

We’ll use **Multinomial Naive Bayes**, which is ideal when features are based on counts or frequencies, like term frequency (TF) or TF-IDF vectors. It pairs perfectly with the bag-of-words approach we used above.

In [106]:
# There are several classes that can be imported, one is multinomial which is for multi-class classification.
from sklearn.naive_bayes import MultinomialNB

In [108]:
# Initialize or create an instance of the classifier
cl = MultinomialNB()
cl

MultinomialNB()

In [110]:
# Now train the classifer, which is incredibly fast
# Use the training set and the dependent variable, y_train
cl.fit(X_train_mat, y_train)

MultinomialNB()

## Make predictions and understand the output

Now, we'll predict every article in the dataset and see how well the model performs.

In [114]:
# The classifier has a method .predict()
# Use the test set matrix and save it as y_pred because it's predicting y's
# So, of course, the output gives us the lables
y_pred = cl.predict(X_test_mat)
y_pred

array(['none', 'Computer Science', 'none', ..., 'none',
       'Computer Science', 'Computer Science'], dtype='<U16')

In [116]:
# Check the shape of y_pred
y_pred.shape

(6292,)

In [118]:
# Can also predict the predicted probabiliites because the raw output is not just
# the labels but the probability for each element of being a given label
# So, to read this output: It's an array of six elements that shows the probability 
# of being in each specific class
cl.predict_proba(X_test_mat)

array([[1.38775606e-041, 1.00000000e+000],
       [9.99999991e-001, 9.22016376e-009],
       [2.08121799e-122, 1.00000000e+000],
       ...,
       [9.96036218e-046, 1.00000000e+000],
       [1.00000000e+000, 1.74213950e-010],
       [1.00000000e+000, 3.10730567e-052]])

In [120]:
# Just to underscore that point, here's just the array for the first article at index 0
# As we'd expect, there are exactly six elements
cl.predict_proba(X_test_mat)[0]

array([1.38775606e-41, 1.00000000e+00])

## Evaluate the model's performance

Now to analyze how well the model predicted the classes. 

Now that we’ve generated predictions, it’s time to evaluate how well the model performed. An easy way to do so is to import classification report as we do below. This allows us to compare the predicted labels ('y_pred') to the *actual* labels. We also get a breakdown of precision, recall, and F1-score for each class -- plus averages across all classes.

In [125]:
from sklearn.metrics import classification_report

In [138]:
# So, the classification_report() function takes as an input y_test, the real class and y_pred the predicted class
# Effectively, this compares the actual labels (y_test) with the predicted labels (y_pred) and summarizes the model's performance
# Use print to clean up the output
print(classification_report(y_test, y_pred))

                  precision    recall  f1-score   support

Computer Science       0.79      0.90      0.85      2575
            none       0.93      0.84      0.88      3717

        accuracy                           0.86      6292
       macro avg       0.86      0.87      0.86      6292
    weighted avg       0.87      0.86      0.87      6292



**Takeaway**: Performance is understandably uneven across the board, but that's to be expected with six classes to predict. As we see in the classification report, the model handles the larger categories, like Physics and Math, quite well. But, when it comes to smaller ones, like Quant Bio and Quant Finance, there are all sorts of problems. What is likely happening here is that there are too few training samples. 